In [ ]:
import os
import scanpy as sc
import celltypist
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import tarfile
import urllib.request
import shutil
import glob
import ssl
import gzip  # <--- Added this to handle the uncompressed files

# Suppress warnings
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

print("Environment setup complete.")

In [ ]:
# --- 1. Setup Data Directory ---
os.makedirs("data", exist_ok=True)

# --- 2. Download Whitelist ---
url = "https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"
whitelist_path = "data/whitelist.txt.gz"

ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

if not os.path.exists(whitelist_path):
    print(f"Downloading whitelist...")
    try:
        with urllib.request.urlopen(url, context=ssl_context) as response, open(whitelist_path, 'wb') as out_file:
            shutil.copyfileobj(response, out_file)
        os.system(f"gunzip -f {whitelist_path}")
        print("Whitelist ready.")
    except Exception as e:
        print(f"Whitelist download failed: {e}")

# --- 3. Extract Input Data ---
tar_filename = "toy_read_ref_set.tar.gz"
if os.path.exists(tar_filename):
    print(f"Extracting {tar_filename}...")
    try:
        with tarfile.open(tar_filename, "r:*") as tar:
            tar.extractall()
        print("Extraction complete.")
    except Exception as e:
        print(f"Extraction failed: {e}")

# --- 4. Find and Rename Files ---
target_genome = "genome.fa"
target_gtf = "genes.gtf"
target_r1 = "r1.fq.gz"
target_r2 = "r2.fq.gz"

def safe_rename(src, dst):
    if os.path.abspath(src) == os.path.abspath(dst): return
    print(f"Renaming {src} -> {dst}")
    shutil.move(src, dst)

def find_file(extensions):
    candidates = []
    for root, _, files in os.walk("."):
        if "data/" in root: continue
        for f in files:
            if f.lower().endswith(extensions) and not f.startswith("._"):
                candidates.append(os.path.join(root, f))
    return candidates

# Find Genome
genomes = find_file(('.fa', '.fasta'))
if genomes: safe_rename(max(genomes, key=os.path.getsize), target_genome)

# Find GTF
gtfs = find_file(('.gtf',))
if gtfs: safe_rename(max(gtfs, key=os.path.getsize), target_gtf)

# Find FASTQs (Look for both compressed and uncompressed)
# Updated to include .fastq and .fq
fastqs = [f for f in find_file(('.fastq.gz', '.fq.gz', '.fastq', '.fq')) if "whitelist" not in f]
fastqs = sorted(fastqs)

def compress_and_move(src, dest):
    if src.endswith('.gz'):
        safe_rename(src, dest)
    else:
        print(f"Compressing {src} to {dest}...")
        with open(src, 'rb') as f_in:
            with gzip.open(dest, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)

if len(fastqs) >= 2:
    compress_and_move(fastqs[0], target_r1)
    compress_and_move(fastqs[1], target_r2)
    print(f"Prepared FASTQs: {target_r1}, {target_r2}")
else:
    print("WARNING: Could not find 2 FASTQ files. Listing all files:")
    os.system("ls -R")

# --- 5. Generate Transcriptome ---
if not os.path.exists("transcripts.fa"):
    print("Installing gffread...")
    os.system("mamba install -y -c bioconda gffread")
    print("Generating transcripts.fa...")
    os.system(f"gffread -w transcripts.fa -g {target_genome} {target_gtf}")

In [ ]:
%%bash
set -e

# --- Configuration ---
TRANSCRIPTOME="transcripts.fa"
GTF="genes.gtf"
R1="r1.fq.gz"
R2="r2.fq.gz"

# Outputs
IDX="data/salmon_index"
MAP_OUT="data/alevin_out"

echo "--- Checking Inputs ---"
ls -lh $TRANSCRIPTOME $GTF $R1 $R2

# --- Step 1: Generate t2g ---
echo "Generating t2g.tsv..."
grep 'transcript_id' $GTF | \
awk -F';' '{print $1, $3}' | \
sed 's/transcript_id "//' | sed 's/"; gene_id "/\t/' | sed 's/"//' > data/t2g.tsv

# --- Step 2: Build Salmon Index (Silent Mode) ---
echo "Building Salmon index (output muted to prevent log overflow)..."
if ! salmon index -t $TRANSCRIPTOME -i $IDX -p 2 > salmon_index.log 2>&1; then
    echo "ERROR: Salmon indexing failed! Log output:"
    cat salmon_index.log
    exit 1
fi
echo "Index built successfully."

# --- Step 3: Run Salmon Alevin (Silent Mode) ---
echo "Running Salmon Alevin (output muted)..."
# ADDED: --chromium flag is required for 10x data
if ! salmon alevin -l ISR -1 $R1 -2 $R2 \
  -i $IDX \
  --tgMap data/t2g.tsv \
  --output $MAP_OUT \
  --chromium \
  --rad --sketch \
  -p 2 > salmon_alevin.log 2>&1; then
    echo "ERROR: Salmon Alevin failed! Log output:"
    cat salmon_alevin.log
    exit 1
fi

echo "Salmon pipeline complete."

In [ ]:
%%bash
set -e
MAP_OUT="data/alevin_out"
QUANT_OUT="data/fry_quant"
WHITELIST="data/whitelist.txt"
T2G="data/t2g.tsv"

echo "Generating permit list..."
alevin-fry generate-permit-list -d forward -i $MAP_OUT -o $QUANT_OUT -u $WHITELIST > fry_permit.log 2>&1

echo "Collating records..."
alevin-fry collate -i $QUANT_OUT -r $MAP_OUT -t 2 > fry_collate.log 2>&1

echo "Quantifying..."
alevin-fry quant -i $QUANT_OUT -o $QUANT_OUT/res -t 2 -r cr-like -m $T2G --use-mtx > fry_quant.log 2>&1

# Compress output for Scanpy
if [ -f "$QUANT_OUT/res/alevin/matrix.mtx" ]; then
    gzip -f "$QUANT_OUT/res/alevin/matrix.mtx"
fi

echo "Quantification complete."

In [ ]:
print("Loading count matrix...")
adata = sc.read_10x_mtx('data/fry_quant/res/alevin', var_names='gene_symbols', cache=True)

# QC
adata.var['mt'] = adata.var_names.str.startswith('MT-') 
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Filter
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.pct_counts_mt < 5, :]

# Normalize & Cluster
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.tl.leiden(adata)

sc.pl.umap(adata, color=['leiden'], title="Leiden Clustering", show=True)

In [ ]:
model = celltypist.models.Model.load(model='Immune_All_Low.pkl')
predictions = celltypist.annotate(adata, model='Immune_All_Low.pkl', majority_voting=True)

adata.obs['cell_type'] = predictions.predicted_labels['predicted_labels']
adata.obs['conf_score'] = predictions.predicted_labels['conf_score']

sc.pl.umap(adata, color=['cell_type'], title="CellTypist Annotation", legend_loc='on data')
print(adata.obs['cell_type'].value_counts())